In [2]:
"""
Simple Agentic AI Workflow
==========================
4-step agent: Plan → Tool Use → Synthesize → Respond

Install dependency:
    pip install anthropic

Run:
    python agentic_workflow.py
"""

import anthropic
import json

# ── Config ────────────────────────────────────────────────────────────────────

with open("config.json", "r") as f:
    config = json.load(f)

API_KEY = config["API_KEY"]
MODEL = config["MODEL"]
client = anthropic.Anthropic(api_key=API_KEY)


# ── Calculator tool (runs locally) ───────────────────────────────────────────

def run_calculator(expression: str) -> str:
    """Safely evaluate a simple math expression."""
    # Only allow digits and basic operators
    safe = "".join(c for c in expression if c in "0123456789+-*/.() %")
    try:
        result = eval(safe)
        return str(round(result, 4))
    except Exception as e:
        return f"Error: {e}"


# ── One reusable Claude call ──────────────────────────────────────────────────

def call_claude(system: str, user_message: str, tools: list = None) -> anthropic.types.Message:
    kwargs = {
        "model":      MODEL,
        "max_tokens": 1000,
        "system":     system,
        "messages":   [{"role": "user", "content": user_message}],
    }
    if tools:
        kwargs["tools"] = tools
    return client.messages.create(**kwargs)


def get_text(response) -> str:
    """Extract plain text from a Claude response."""
    return "\n".join(
        block.text for block in response.content if block.type == "text"
    )

def get_tool_use(response):
    """Return the first tool_use block, or None."""
    return next(
        (block for block in response.content if block.type == "tool_use"),
        None
    )


# ── Step helpers ──────────────────────────────────────────────────────────────

def print_step(number: int, title: str, content: str):
    width = 60
    print(f"\n{'─' * width}")
    print(f"  Step {number}: {title}")
    print(f"{'─' * width}")
    print(content)


# ── Define the calculator tool for Claude ────────────────────────────────────

CALCULATOR_TOOL = {
    "name": "calculator",
    "description": "Evaluates a math expression and returns the numeric result. Use for any arithmetic.",
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": 'A math expression like "142 * 37" or "100 / 7"'
            }
        },
        "required": ["expression"]
    }
}


# ── Main agent loop ───────────────────────────────────────────────────────────

def run_agent(task: str):
    print(f"\n{'═' * 60}")
    print(f"  Task: {task}")
    print(f"{'═' * 60}")

    tool_result = None

    # ── Step 1: Plan ──────────────────────────────────────────────────────────
    plan_response = call_claude(
        system="""You are a planning agent. Given a user task, briefly identify:
1) What the user wants
2) Whether a calculator tool will be needed
3) What the final response should include
Be very concise — 3 sentences max.""",
        user_message=task
    )
    plan = get_text(plan_response)
    print_step(1, "Plan — understand the task", plan)

    # ── Step 2: Tool use ──────────────────────────────────────────────────────
    tool_response = call_claude(
        system="You are a tool-use agent. If the task requires arithmetic, call the calculator tool. Otherwise do nothing.",
        user_message=task,
        tools=[CALCULATOR_TOOL]
    )

    tool_use_block = get_tool_use(tool_response)

    if tool_use_block:
        expression = tool_use_block.input["expression"]
        tool_result = run_calculator(expression)
        content = f'Called calculator("{expression}")\n→ Result: {tool_result}'
    else:
        content = "No calculator needed — skipped."

    print_step(2, "Tool use — calculator", content)

    # ── Step 3: Synthesize ────────────────────────────────────────────────────
    synth_context = f"Task: {task}\n\nPlan: {plan}\n\n"
    synth_context += (
        f"Tool result: calculator returned {tool_result}"
        if tool_result else "No tool was used."
    )

    synth_response = call_claude(
        system="You are a synthesis agent. Given a task, a plan, and any tool results, summarize what you now know and what the final answer should cover. Be brief (2–3 sentences).",
        user_message=synth_context
    )
    synth = get_text(synth_response)
    print_step(3, "Synthesize — combine results", synth)

    # ── Step 4: Final response ────────────────────────────────────────────────
    final_context = f"Task: {task}\n\nSynthesis: {synth}"
    if tool_result:
        final_context += f"\n\nCalculated value: {tool_result}"

    final_response = call_claude(
        system="You are a response agent. Write a clear, friendly, complete response to the user based on the synthesis and any tool results provided.",
        user_message=final_context
    )
    final = get_text(final_response)
    print_step(4, "Final response", final)

    print(f"\n{'═' * 60}\n")
    return final


# ── Run it ────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    # Change this to any task you like
    task = "Calculate 142 × 37, then tell me a fun fact about that number."
    run_agent(task)


════════════════════════════════════════════════════════════
  Task: Calculate 142 × 37, then tell me a fun fact about that number.
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
  Step 1: Plan — understand the task
────────────────────────────────────────────────────────────
**Plan:**
1. The user wants the product of 142 × 37 and a fun fact about the result.
2. Yes, a calculator tool is needed to compute 142 × 37.
3. The final response should include the calculated result (5,254) and an interesting fun fact about that number.

---

**142 × 37 = 5,254**

🎉 **Fun fact about 5,254:**
5,254 is an **even composite number** — it breaks down into prime factors as **2 × 2,627**, and interestingly, 2,627 is itself a prime number! It's also the equivalent of roughly the number of **weeks in just over 100 years** (100 years ≈ 5,200 weeks), so 5,254 weeks would take you about 100 years and 54 weeks into the future! 🕰️

─

In [4]:
# Build a test dataset like this:
test_cases = [
    {
        "task": "Calculate 142 × 37, then tell me a fun fact about that number.",
        "expected_answer": 5254,        # for exact match
        "requires_tool": True,
    },
    {
        "task": "What are 3 steps to learn Python?",
        "expected_answer": None,        # no exact answer, use LLM judge
        "requires_tool": False,
    },
]

for t in test_cases:
    task = t['task']
    run_agent(task)


════════════════════════════════════════════════════════════
  Task: Calculate 142 × 37, then tell me a fun fact about that number.
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
  Step 1: Plan — understand the task
────────────────────────────────────────────────────────────
**Planning:**
The user wants the product of 142 × 37 and a fun fact about the result. A calculator tool will be needed to compute the multiplication. The final response should include the calculated result (5,254) and an interesting fun fact about that number.

---

**142 × 37 = 5,254**

Here's a fun fact about **5,254**:
5,254 is an **even composite number** — it breaks down into prime factors as **2 × 2,627**, and interestingly, 2,627 is itself prime! It's also the equivalent of roughly the number of **weeks in just over 100 years** (100 years ≈ 5,200 weeks), so 5,254 weeks would be about **100 years and 1 month** of living — quite a mi

In [9]:
import json
import re

def judge(task, final_response):
    response = call_claude(
        system="""You are an evaluator. Score the response 1-5 on:
- Correctness
- Completeness
- Clarity
Return JSON only: {"score": N, "reason": "..."}""",
        user_message=f"Task: {task}\n\nResponse: {final_response}"
    )
    raw = get_text(response)
    print("DEBUG raw:", repr(raw))      # <-- add this to see exactly what came back

    # use regex to pull out the JSON object, regardless of surrounding text
    match = re.search(r'\{.*?\}', raw, re.DOTALL)
    if not match:
        return {"score": None, "reason": f"Could not parse response: {raw}"}
    
    return json.loads(match.group())

for t in test_cases:
    task = t["task"]
    final = run_agent(task)                   # run the agent, get the string answer
    score = judge(task, final)                # evaluate it
    results.append({"task": task, "score": score})
    print(f"\nTask:   {task}")
    print(f"Score:  {score['score']}/5")
    print(f"Reason: {score['reason']}")


════════════════════════════════════════════════════════════
  Task: Calculate 142 × 37, then tell me a fun fact about that number.
════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
  Step 1: Plan — understand the task
────────────────────────────────────────────────────────────
**Plan:**
1. The user wants the product of 142 × 37 and a fun fact about the result.
2. Yes, a calculator tool is needed to compute 142 × 37.
3. The final response should include the calculated result (5,254) and an interesting fun fact about that number.

---

**142 × 37 = 5,254**

🎉 **Fun fact about 5,254:**
5,254 is an **even composite number** with multiple factors (1, 2, 7, 14, 376, 751, and more… wait, let me be precise):

The factors of 5,254 are: **1, 2, 11, 22, 239, 478, 2,627, and 5,254** — making it the product of 2 × 11 × 239.

Also, 5,254 falls between two well-known milestones — **5,280 feet = 1 mile**, so 5,254 is just **26